In [ ]:
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启内核就能用上新代码
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, viz
from signal_insight.viz import MAIN, MAIN_MID, MAIN_LIGHT, AMBER, GRAY, GRAY_DARK, RED, INK, MUTED, FAINT
viz.apply_theme()
con = mysql.connect(config.DB_SCHEMA)          # signal.db 已迁移到 MySQL 的 signal_insight 库
DAY = config.ANALYSIS_DAY                      # 2018-10-03
NB = "E01_每小时定位与三商圈统计"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"
pd.set_option("display.max_columns", 20)

In [ ]:
# ── 本实验的口径（先写清楚，再写代码）──
# 1) 一个人一小时一个位置：每个 (imsi, hour) 一行
# 2) 如果这一小时有停留覆盖 ≥ 30 分钟：取覆盖最长的那个停留的位置（并列取最早出现的那一段）
# 3) 否则：取这个人这一小时所有信令位置的中位数
# 4) 这一小时一条信令都没有：不可定位，不硬补；占比的分母是「当小时可定位人数」
settings = config.load_settings()
K = settings["hourly"]["min_cover_minutes"]      # 30 分钟
RADIUS = settings["hotspot"]["radius_m"]         # 500 米，西塔的圆

stay = mysql.read_frame("SELECT * FROM dws_stay", con, parse_dates=["start_time", "end_time"])
sig = mysql.read_frame(
    "SELECT s.imsi, s.ts, c.lng, c.lat FROM dwd_signal s "
    "JOIN dim_cell c ON s.lac_cell = c.lac_cell", con, parse_dates=["ts"])
hot = mysql.read_frame("SELECT * FROM dim_hotspot", con).iloc[0]
aoi = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", con), "name")
print(f"停留 {len(stay):,} 段；信令 {len(sig):,} 条；热点 {hot['name']}（{RADIUS} 米）；商圈 {len(aoi)} 个")

In [ ]:
# ── 步骤 1：一段停留覆盖了第 h 小时多少分钟 ──
# 停留的起止时刻换成「当天第几小时（小数）」，再和第 h 小时的 [h, h+1] 求交集
d0 = pd.Timestamp(DAY)
start_h = (stay["start_time"] - d0).dt.total_seconds() / 3600
end_h = (stay["end_time"] - d0).dt.total_seconds() / 3600

parts = []
for h in range(24):
    cover_min = (np.minimum(end_h, h + 1) - np.maximum(start_h, h)).clip(lower=0) * 60
    ok = cover_min >= K                     # 只有覆盖满 30 分钟才有资格用停留定位
    parts.append(stay.loc[ok, ["imsi", "lng", "lat", "start_time"]]
                 .assign(hour=h, cover_min=cover_min[ok]))
coverage = pd.concat(parts, ignore_index=True)
print(f"有停留覆盖的小时：{len(coverage):,} 个（人 × 小时）")
coverage.head()

In [ ]:
# ── 步骤 2：同一小时被多段停留覆盖时，取覆盖最长的那一段 ──
# 并列（覆盖分钟数相同）时取最早开始的那一段——排序里 cover_min 降序、start_time 升序，
# 再每组取第一行，就同时满足这两条规则
by_stay = (coverage.sort_values(["imsi", "hour", "cover_min", "start_time"],
                                ascending=[True, True, False, True])
           .drop_duplicates(["imsi", "hour"])
           .assign(src="停留"))
print(f"用停留定位：{len(by_stay):,} 个人×小时")
by_stay.head()

In [ ]:
# ── 步骤 3：没有停留覆盖的小时，用该小时信令位置的中位数 ──
# 取中位数而不是平均数：一小时内几条信令可能分散在几个基站，平均数会落到它们中间的空地上
# （甚至落到河里），中位数一定落在真实出现过的位置附近
by_sig = (sig.assign(hour=sig["ts"].dt.hour)
          .groupby(["imsi", "hour"], as_index=False)[["lng", "lat"]].median())
by_sig = by_sig.merge(by_stay[["imsi", "hour"]], how="left", indicator=True)
by_sig = by_sig[by_sig["_merge"] == "left_only"].drop(columns="_merge").assign(src="信令")
print(f"用信令中位数定位：{len(by_sig):,} 个人×小时")
print(f"这个小时一条信令都没有的人不会出现在这里——不可定位，不硬补")

In [ ]:
# ── 步骤 4：两档合并，得到「每人每小时一个位置」 ──
uh = pd.concat([by_stay[["imsi", "hour", "lng", "lat", "src"]],
                by_sig[["imsi", "hour", "lng", "lat", "src"]]], ignore_index=True)
uh = uh.sort_values(["imsi", "hour"], ignore_index=True)
print(f"合计 {len(uh):,} 行：停留定位 {int((uh['src'] == '停留').sum()):,}、信令定位 {int((uh['src'] == '信令').sum()):,}")
print(f"涉及用户 {uh['imsi'].nunique():,} 人；每人平均 {len(uh) / uh['imsi'].nunique():.1f} 个小时可定位")
uh.head()

In [ ]:
# ── 对账 1：和流水线算出的 dws_user_hour 逐行比 ──
lib = mysql.read_frame("SELECT * FROM dws_user_hour", con).sort_values(["imsi", "hour"], ignore_index=True)
merged = uh.merge(lib[["imsi", "hour", "lng", "lat", "src"]], on=["imsi", "hour"], suffixes=("", "_lib"))
delta = merged[["lng", "lat"]].to_numpy() - merged[["lng_lib", "lat_lib"]].to_numpy()
assert len(uh) == len(lib) == 102_859, (len(uh), len(lib))
assert np.abs(delta).max() < 1e-9, np.abs(delta).max()
assert (merged["src"] == merged["src_lib"]).all()
print(f"✅ 对账通过：与库里 dws_user_hour 逐行一致（{len(uh):,} 行，坐标最大差 {np.abs(delta).max():.1e}）")

In [ ]:
# ── 步骤 5：给每个位置定区域 ──
# 西塔是圆（离中心 ≤ 500 米）；太原街、中街是多边形（点在面内）
# 区域可以重叠：西塔的圆和中街/太原街的边界有交集，所以下面各区域人数不能相加
uh["dist_xita_m"] = geo.haversine(uh["lat"], uh["lng"], hot["lat"], hot["lng"])
uh["in_xita"] = uh["dist_xita_m"] <= RADIUS
uh["aoi"] = geo.point_in_polygons(uh["lng"], uh["lat"], aoi)
uh[["imsi", "hour", "src", "dist_xita_m", "in_xita", "aoi"]].head()

In [ ]:
# ── 步骤 6：逐小时统计三个区域的人数与占比 ──
AREAS = ["西塔", "太原街商圈", "中街商圈"]
locatable = uh.groupby("hour").size().reindex(range(24), fill_value=0).rename("可定位")
counts = pd.DataFrame({"可定位": locatable})
counts["西塔"] = uh[uh["in_xita"]].groupby("hour").size().reindex(range(24), fill_value=0)
counts["太原街商圈"] = uh[uh["aoi"] == "太原街商圈"].groupby("hour").size().reindex(range(24), fill_value=0)
counts["中街商圈"] = uh[uh["aoi"] == "中街商圈"].groupby("hour").size().reindex(range(24), fill_value=0)
for a in AREAS:
    counts[a + "占比"] = (counts[a] / counts["可定位"]).round(4)
counts

In [ ]:
# ── 出图 E01_01：形状与峰值 ──
# 三个区域人数差近 7 倍，直接画三条绝对值的线，中街会被压成贴着横轴的一条直线；
# 所以左边归一化（除以各自当天峰值）看形状，右边只画各自峰值时刻的绝对值
COLOR = {"西塔": MAIN, "太原街商圈": MAIN_MID, "中街商圈": AMBER}
peaks = {a: (int(counts[a].idxmax()), int(counts[a].max())) for a in AREAS}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.6), layout="constrained")
for a in AREAS:
    ax1.plot(counts.index, counts[a] / counts[a].max(), marker="o", ms=3, color=COLOR[a], label=a)
ax1.axvline(19, color=FAINT, lw=0.8, ls="--")
ax1.set_xlabel("小时"); ax1.set_ylabel("占当天峰值"); ax1.set_ylim(0, 1.08)
ax1.grid(axis="y"); ax1.legend(frameon=False, fontsize=9)
bars = ax2.bar([f"{a}\n{peaks[a][0]} 点" for a in AREAS],
               [peaks[a][1] for a in AREAS], color=[COLOR[a] for a in AREAS], width=0.55)
ax2.bar_label(bars, padding=3, fontsize=10)
ax2.set_ylabel("峰值时刻的人数"); ax2.grid(axis="y")
viz.finish(fig, "三个商圈的日内节律形状相近，绝对量相差近 7 倍",
           "左：每小时人数除以各自当日峰值（只看形状）；右：各自峰值时刻的人数。口径：每人每小时一个位置；区域可重叠，人数不可相加",
           SOURCE, "E01_01_三商圈节律.png")

In [ ]:
# ── 出图 E01_02：占比 ──
# 占比的分母是「当小时可定位人数」：这一小时一条信令都没有的人不计入分母，也不计为 0
fig, ax = plt.subplots(figsize=(10, 4.6))
for a in AREAS:
    ax.plot(counts.index, counts[a + "占比"], marker="o", ms=3, color=COLOR[a], label=a)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("小时"); ax.set_ylabel("占当小时可定位人数")
ax.grid(axis="y"); ax.legend(frameon=False)
viz.finish(fig, f"西塔占比在 19 点达到 {counts.loc[19, '西塔占比']:.0%}，另两个商圈全天不到 10%",
           "分母 = 当小时可定位人数；三条线都是百分比、单位相同，可以放在同一个纵轴上",
           SOURCE, "E01_02_三商圈占比.png")

In [ ]:
# ── 对账 2：与人人都能复算的几个数对齐 ──
assert (peaks["西塔"], peaks["太原街商圈"], peaks["中街商圈"]) == ((19, 2_037), (17, 448), (16, 297))
assert int(counts.loc[19, "可定位"]) == 5_069
assert int(counts.loc[19, "西塔"]) == 2_037
assert str(counts["西塔占比"].idxmax()) == "19"
print(f"✅ 对账通过：西塔 19 点峰值 2,037 人（分母 5,069，占 {counts.loc[19, '西塔占比']:.0%}）；"
      f"太原街 17 点 {peaks['太原街商圈'][1]} 人；中街 16 点 {peaks['中街商圈'][1]} 人")

In [ ]:
# ── 小结 ──
# 1) 三档规则完整覆盖了「可定位」的人：有停留用停留、没停留用信令中位数、都没有就不补
# 2) 停留定位 {:.1%}、信令定位 {:.1%}，前者占多数说明 30 分钟覆盖阈值选得合理
# 3) 西塔的峰值出现在 19 点，与「傍晚客流」的直觉一致；太原街、中街峰值更早（17、16 点）
# 4) 区域可以重叠，所以三列相加不等于可定位人数，占比也不能相加
sto = (uh["src"] == "停留").mean()
print(f"停留定位 {sto:.1%}，信令定位 {1 - sto:.1%}")
print(f"西塔占比从 0 点的 {counts.loc[0, '西塔占比']:.0%} 升到 19 点的 {counts.loc[19, '西塔占比']:.0%}，"
      f"再回落到 23 点的 {counts.loc[23, '西塔占比']:.0%}")
con.close()